In [186]:

import os
import matplotlib.pyplot as plt 
import kagglehub
import pandas as pd
import shutil
from ultralytics import YOLO
import torch
from torchvision import transforms
from PIL import Image
import random

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [187]:
# Remove the existing processed dataset
if os.path.exists(processed_path):
    shutil.rmtree(processed_path)
    print("Existing processed_dataset removed!")

# Recreate an empty destination folde

In [188]:
import os
import hashlib

classes = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]


def get_image_hash(image_path):
    with open(image_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def clean_dataset():
    seen_pairs = set()

    missing_labels = 0
    orphan_labels = 0
    duplicates_removed = 0

    for split in ["Train", "Val"]:
        for class_name in classes:

            images_path = os.path.join(
                path, split, class_name, "images"
            )
            labels_path = os.path.join(
                path, split, class_name, "labels"
            )

            # Image filenames without extensions
            image_names = {
                os.path.splitext(f)[0]
                for f in os.listdir(images_path)
                if os.path.isfile(os.path.join(images_path, f))
            }

            # Remove orphan labels
            for label in os.listdir(labels_path):
                label_path = os.path.join(labels_path, label)

                if (
                    os.path.isfile(label_path)
                    and label.endswith(".txt")
                    and os.path.splitext(label)[0] not in image_names
                ):
                    os.remove(label_path)
                    orphan_labels += 1

            # Check images and matching labels
            for image in os.listdir(images_path):
                image_path = os.path.join(images_path, image)

                if not os.path.isfile(image_path):
                    continue

                label = os.path.splitext(image)[0] + ".txt"
                label_path = os.path.join(labels_path, label)

                # Remove images without labels
                if not os.path.isfile(label_path):
                    os.remove(image_path)
                    missing_labels += 1
                    continue

                # Hash image and read its annotation
                image_hash = get_image_hash(image_path)

                with open(label_path, "r") as f:
                    label_content = f.read()

                pair = (image_hash, label_content)

                # Remove duplicate image-label pairs
                if pair in seen_pairs:
                    os.remove(image_path)
                    os.remove(label_path)
                    duplicates_removed += 1
                    continue

                seen_pairs.add(pair)

    print("Duplicate pairs removed:", duplicates_removed)
    print("Images without labels removed:", missing_labels)
    print("Orphan labels removed:", orphan_labels)


clean_dataset()

Duplicate pairs removed: 0
Images without labels removed: 0
Orphan labels removed: 0


In [189]:

# a function that display a folder tree
import os

def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for f in files[:3]:
            print('{}{}'.format(subindent, f))
            
list_files(path)


dataset/
    Train/
        Glioma/
            images/
                Te-glTr_0000.jpg
                Te-glTr_0005.jpg
                Te-glTr_0007.jpg
            labels/
                Te-glTr_0000.txt
                Te-glTr_0005.txt
                Te-glTr_0007.txt
        Meningioma/
            images/
                Tr-meTr_0008.jpg
                Tr-me_0010.jpg
                Tr-me_0011.jpg
            labels/
                Tr-meTr_0008.txt
                Tr-me_0010.txt
                Tr-me_0011.txt
        No Tumor/
            images/
                1.jpg
                3.jpg
                4.jpg
            labels/
                1.txt
                3.txt
                4.txt
        Pituitary/
            images/
                Te-pi_0010.jpg
                Te-pi_0011.jpg
                Te-pi_0013.jpg
            labels/
                Te-pi_0010.txt
                Te-pi_0011.txt
                Te-pi_0013.txt
    Val/
        Glioma/
            imag